# Retail Sales Forecasting — Walmart Weekly Sales

Predict weekly sales for each store-department, compare models, and export results for a Power BI dashboard and a Streamlit app.

**Files needed (from Kaggle, "Walmart Recruiting – Store Sales Forecasting"):** `train.csv`, `stores.csv`, `features.csv`.
Upload them using the folder icon on the left before running.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor          # pre-installed in Google Colab

## 2. Load the data

In [ ]:
train = pd.read_csv("/content/train.csv")        # weekly sales per store and department
stores = pd.read_csv("/content/stores.csv")      # store type and size
features = pd.read_csv("/content/features.csv")  # temperature, fuel price, markdowns, CPI, unemployment

print(train.shape, stores.shape, features.shape)
train.head()

## 3. Clean and merge

In [ ]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

# features.csv also has IsHoliday; drop it so the column isn't duplicated
features = features.drop(columns=["IsHoliday"])

df = train.merge(stores, on="Store", how="left")                 # add Type and Size
df = df.merge(features, on=["Store", "Date"], how="left")        # add external factors

# Markdowns (promotions) are missing when there was no promotion -> treat as 0
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
df[markdown_cols] = df[markdown_cols].fillna(0)

# Fill any remaining gaps in CPI / Unemployment with the store's previous value
df = df.sort_values(["Store", "Date"])
df[["CPI", "Unemployment"]] = df.groupby("Store")[["CPI", "Unemployment"]].ffill()
df[["CPI", "Unemployment"]] = df[["CPI", "Unemployment"]].fillna(df[["CPI", "Unemployment"]].median())

df = df.drop_duplicates()
print(df.isnull().sum())

## 4. Exploration (EDA)

In [ ]:
print("Rows:", len(df), "| Stores:", df["Store"].nunique(), "| Departments:", df["Dept"].nunique())
print("Negative sales rows (returns):", (df["Weekly_Sales"] < 0).sum())

# Total sales over time - shows seasonality and holiday peaks
df.groupby("Date")["Weekly_Sales"].sum().plot(figsize=(10, 4), title="Total weekly sales (all stores)")
plt.ylabel("Sales ($)")
plt.show()

# Holiday vs normal weeks
h = df.groupby("IsHoliday")["Weekly_Sales"].mean()
print(f"Holiday weeks sell {(h[True] / h[False] - 1) * 100:.1f}% more on average")

# Average sales by store type
df.groupby("Type")["Weekly_Sales"].mean().plot(kind="bar", title="Average weekly sales by store type")
plt.ylabel("Sales ($)")
plt.show()

## 5. Feature engineering

In [ ]:
# Calendar features: help the model learn seasonality
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["IsHoliday"] = df["IsHoliday"].astype(int)

# Keep a readable store type for the dashboard, then one-hot encode it for the models
df["StoreType"] = df["Type"]
df = pd.get_dummies(df, columns=["Type"], drop_first=True, dtype=int)   # creates Type_B, Type_C

# Lag features: past sales of the same store-department (strongest forecasting signals)
df = df.sort_values(["Store", "Dept", "Date"])
g = df.groupby(["Store", "Dept"])["Weekly_Sales"]
df["lag_1"] = g.shift(1)                                                # last week's sales
df["lag_52"] = g.shift(52)                                              # same week last year
df["roll_4"] = g.transform(lambda s: s.shift(1).rolling(4).mean())      # average of the previous 4 weeks

# Rows without full history (mostly the first year) can't have lag values -> drop them
df = df.dropna(subset=["lag_1", "lag_52", "roll_4"]).reset_index(drop=True)
print("Rows after creating lag features:", len(df))

## 6. Time-based train/test split
Train on the older 80% of weeks, test on the most recent 20%, so the model never sees the future (avoids data leakage).

In [ ]:
features_list = [
    "Store", "Dept", "Year", "Month", "Week", "IsHoliday", "Size", "Type_B", "Type_C",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "lag_1", "lag_52", "roll_4",
]

dates = np.sort(df["Date"].unique())
cutoff = dates[int(len(dates) * 0.8)]

train_df = df[df["Date"] < cutoff]
test_df = df[df["Date"] >= cutoff]

X_train, y_train = train_df[features_list], train_df["Weekly_Sales"]
X_test, y_test = test_df[features_list], test_df["Weekly_Sales"]
print("Train until:", pd.Timestamp(cutoff).date(), "| train rows:", len(train_df), "| test rows:", len(test_df))

## 7. Evaluation helpers
WMAE gives holiday weeks 5x weight (the metric used in the original Kaggle competition).

In [ ]:
def evaluate(name, actual, pred, is_holiday):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    weights = np.where(is_holiday == 1, 5, 1)                     # holiday weeks count 5x
    wmae = np.sum(weights * np.abs(actual - pred)) / np.sum(weights)
    return {"Model": name, "MAE": round(mae), "RMSE": round(rmse), "WMAE": round(wmae)}

## 8. Train and compare models

In [ ]:
results = []

# Baseline: each store-department's average sales in the training period
avg = train_df.groupby(["Store", "Dept"])["Weekly_Sales"].mean().rename("base")
base_pred = test_df.join(avg, on=["Store", "Dept"])["base"].fillna(y_train.mean())
results.append(evaluate("Baseline (store-dept average)", y_test, base_pred, test_df["IsHoliday"]))

# Linear Regression
lr = LinearRegression().fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results.append(evaluate("Linear Regression", y_test, lr_pred, test_df["IsHoliday"]))

# Random Forest (takes a few minutes)
rf = RandomForestRegressor(n_estimators=50, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results.append(evaluate("Random Forest", y_test, rf_pred, test_df["IsHoliday"]))

# XGBoost (gradient boosting: trees built one after another, each fixing earlier mistakes)
xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
results.append(evaluate("XGBoost", y_test, xgb_pred, test_df["IsHoliday"]))

results_df = pd.DataFrame(results)
base_mae = results_df.loc[0, "MAE"]
results_df["MAE reduction vs baseline (%)"] = ((1 - results_df["MAE"] / base_mae) * 100).round(1)   # negative = worse than baseline
results_df

## 9. Time-series cross-validation
Check that results are stable across several time periods, not just one split.

In [ ]:
cv_rows = []
for start_q, end_q in [(0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.0)]:
    start = dates[int(len(dates) * start_q)]
    end = dates[min(int(len(dates) * end_q), len(dates) - 1)]
    tr = df[df["Date"] < start]                                     # train on everything before the window
    te = df[(df["Date"] >= start) & (df["Date"] <= end)]            # test on the next window

    model = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8,
                         subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
    model.fit(tr[features_list], tr["Weekly_Sales"])
    p = model.predict(te[features_list])

    fold_avg = tr.groupby(["Store", "Dept"])["Weekly_Sales"].mean().rename("base")
    b = te.join(fold_avg, on=["Store", "Dept"])["base"].fillna(tr["Weekly_Sales"].mean())

    cv_rows.append({"Test window": f"{pd.Timestamp(start).date()} to {pd.Timestamp(end).date()}",
                    "Baseline MAE": round(mean_absolute_error(te["Weekly_Sales"], b)),
                    "XGBoost MAE": round(mean_absolute_error(te["Weekly_Sales"], p))})

cv_df = pd.DataFrame(cv_rows)
print("Average XGBoost MAE across folds:", round(cv_df["XGBoost MAE"].mean()))
cv_df

## 10. Feature importance

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=features_list).sort_values()
importance.plot(kind="barh", figsize=(8, 7), title="XGBoost feature importance")
plt.show()

## 11. Actual vs predicted (test period)

In [ ]:
best_name = results_df.iloc[1:].sort_values("MAE").iloc[0]["Model"]          # best ML model by MAE
best_pred = {"Linear Regression": lr_pred, "Random Forest": rf_pred, "XGBoost": xgb_pred}[best_name]
print("Best model:", best_name)

plot = test_df[["Date"]].copy()
plot["Actual"] = y_test.values
plot["Predicted"] = best_pred
plot.groupby("Date").sum().plot(figsize=(10, 4), title=f"Actual vs predicted weekly sales ({best_name})")
plt.ylabel("Total weekly sales ($)")
plt.show()

## 12. Export predictions for Power BI and Streamlit

In [ ]:
out = test_df[["Store", "Dept", "Date", "StoreType", "Size", "IsHoliday"]].copy()
out["Actual"] = y_test.values
out["Predicted"] = best_pred.round(2)
out["Error"] = (out["Actual"] - out["Predicted"]).round(2)
out["AbsError"] = out["Error"].abs()
out.to_csv("predictions.csv", index=False)

results_df.to_csv("model_results.csv", index=False)

from google.colab import files       # downloads both files to your computer
files.download("predictions.csv")
files.download("model_results.csv")